# 04 · Desafio de Analytics — Camada Gold

Respostas às 6 perguntas de negócio usando **apenas** as tabelas do Star Schema e `display()`.

> **Data limite (perguntas 5 e 6):** a data de lançamento **mais recente** entre filmes com status *Lançado* e `data_lancamento <= hoje` (ignora datas futuras e não lançados). Os recortes são `[limite − 24 meses, limite]` e `[limite − 60 meses, limite]`.

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "workspace", "Catálogo (Unity Catalog)")
spark.sql(f"USE CATALOG `{dbutils.widgets.get('catalogo')}`")

DataFrame[]

### 1. Receita total (R$) somada de todos os filmes da base

In [0]:
display(spark.sql("""
    SELECT CAST(SUM(receita_brl) AS DECIMAL(24,2)) AS receita_total_brl
    FROM gold.fact_movies_performance
"""))

receita_total_brl
870178068274.31


### 2. Top 5 filmes com maior popularidade

In [0]:
display(spark.sql("""
    SELECT m.titulo, f.popularidade
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
Battipaglia 1969,1969.0


### 3. Quantidade de filmes por gênero (do maior para o menor)

In [0]:
display(spark.sql("""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM gold.bridge_movie_genre b
    JOIN gold.dim_genres g ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC, g.nome_genero
"""))

nome_genero,qtd_filmes
Drama,32127
Documentary,18927
Comedy,18537
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
TV Movie,4066


### 4. Top 10 filmes por receita (US$ e R$) com `RANK()`

In [0]:
display(spark.sql("""
    SELECT titulo, receita_usd, receita_brl, posicao_ranking
    FROM (
        SELECT m.titulo, f.receita_usd, f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking
        FROM gold.fact_movies_performance f
        JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    ) t
    WHERE posicao_ranking <= 10          -- RANK() preserva empates: nenhum filme do top 10 fica de fora
    ORDER BY posicao_ranking, titulo
"""))

titulo,receita_usd,receita_brl,posicao_ranking
Avengers: Endgame,2800000000.00,15011640000.00,1
Avatar: The Way of Water,2320250281.00,12439557831.53,2
AVENGERS: INFINITY WAR,2052415039.00,11003612748.59,3
spider-man: no way home,1921847111.00,10303598916.20,4
The Lion King,1663075401.00,8916246147.38,5
Top Gun: Maverick,1488732821.00,7981543273.23,6
Barbie,1428545028.00,7658858458.62,7
The Super Mario Bros. Movie,1355725263.00,7268449852.52,8
Black Panther,1349926083.00,7237358708.79,9
Star Wars: The Last Jedi,1332698830.00,7144998237.28,10


### Data limite para as perguntas 5 e 6

In [0]:
limite = spark.sql("""
    SELECT MAX(data_lancamento) AS limite
    FROM gold.dim_movies
    WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
""").first()["limite"]
print("Data limite superior (lançamento mais recente, sem datas futuras):", limite)

Data limite superior (lançamento mais recente, sem datas futuras): 2026-02-19


### 5. Ator com mais participações em filmes lançados nos últimos 2 anos

In [0]:
display(spark.sql(f"""
    SELECT p.nome_pessoa AS ator,
           COUNT(DISTINCT m.sk_movie_id) AS qtd_participacoes
    FROM gold.dim_movies m
    JOIN gold.bridge_movie_person b ON b.sk_movie_id = m.sk_movie_id
    JOIN gold.dim_people p          ON p.sk_person_id = b.sk_person_id
    WHERE p.tipo_pessoa = 'Ator'
      AND m.status_filme = 'Lançado'
      AND m.data_lancamento BETWEEN add_months(DATE '{limite}', -24) AND DATE '{limite}'
    GROUP BY p.nome_pessoa
    ORDER BY qtd_participacoes DESC, ator
    LIMIT 10
"""))
# A 1ª linha é a resposta; as demais mostram o contexto (e eventuais empates).

ator,qtd_participacoes
Kevin Hart,64
Adlih Torres,12
Andy Dubitsky,12
Cooper Tomlinson,12
Gloria Karel,12
John Simmonds,12
Jonnathon Cripple,12
Sterling L. Pope,12
Tristan Welsh,12
Adriana Asti,6


### 6. Produtora com maior lucro nos últimos 5 anos
Lucro somado por produtora considerando apenas filmes lançados no período e com lucro calculável (receita **e** orçamento conhecidos).

In [0]:
display(spark.sql(f"""
    SELECT c.nome_produtora,
           COUNT(DISTINCT m.sk_movie_id)        AS qtd_filmes_com_lucro,
           SUM(f.lucro_usd)                     AS lucro_total_usd,
           SUM(f.lucro_brl)                     AS lucro_total_brl
    FROM gold.dim_movies m
    JOIN gold.fact_movies_performance f ON f.sk_movie_id = m.sk_movie_id
    JOIN gold.bridge_movie_company b    ON b.sk_movie_id = m.sk_movie_id
    JOIN gold.dim_companies c           ON c.sk_company_id = b.sk_company_id
    WHERE m.status_filme = 'Lançado'
      AND m.data_lancamento BETWEEN add_months(DATE '{limite}', -60) AND DATE '{limite}'
      AND f.lucro_usd IS NOT NULL
    GROUP BY c.nome_produtora
    ORDER BY lucro_total_usd DESC
    LIMIT 10
"""))
# A 1ª linha é a resposta.

nome_produtora,qtd_filmes_com_lucro,lucro_total_usd,lucro_total_brl
Universal Pictures,24,5772329679.00,30947191108.00
Marvel Studios,8,4953462823.00,26557000232.95
Columbia Pictures,12,3662050755.00,19633352712.78
Pascal Pictures,3,2701952454.00,14485977691.63
Illumination,3,2431353473.00,13035215374.79
Paramount,14,2239394101.00,12006063593.68
Kevin Feige Productions,4,2138205367.00,11463560434.10
20th Century Studios,8,2074815383.00,11123707712.89
Lightstorm Entertainment,1,1860250281.00,9973359831.53
Heyday Films,2,1490495872.00,7990995518.56
